<a href="https://colab.research.google.com/github/Lawson-Dong/ESINDy_infra/blob/main/Stacking_ESINDy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
We use K-Fold cross-validation to split the data into 5 folds, each time using 80% for training and 20% for validation.
For each fold, we train 10 base models on that 80% training subset, resulting in a total of 50 base models — each base model is trained on 80% of the full dataset.
After training, the entire dataset is fed to all 50 base models, and their predictions are used as the training set for a meta-learner, which learns the weights α.
"""

import numpy as np
from scipy.signal import savgol_filter
from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import KFold

class StackingSINDy:
    """
    Stacking Ensemble SINDy with 5-fold CV, 10 base models per fold (50 total),
    using 50-dimensional meta-features from predictions on the full training set.
    """
    def __init__(self, poly_degree=2, threshold=0.15, n_folds=5, n_models_per_fold=10,
                 meta_alpha=1.0, sg_window=11, sg_order=3, use_reduced_features=False):
        self.poly_degree = poly_degree
        self.threshold = threshold
        self.n_folds = n_folds                # K
        self.n_models_per_fold = n_models_per_fold  # M
        self.n_estimators = n_folds * n_models_per_fold   # K*M
        self.meta_alpha = meta_alpha
        self.sg_window = sg_window
        self.sg_order = sg_order
        self.use_reduced_features = use_reduced_features

        # Will be set during fit
        self.base_models = []          # list of (n_states, n_features) coefficient arrays
        self.meta_models = []          # list of Ridge models (one per state)
        self.feature_names = None
        self.poly = None

    def _compute_derivative_sg(self, X, dt):
        """Smooth derivatives with Savitzky-Golay filter."""
        n_samples, n_states = X.shape
        dX = np.zeros_like(X)
        for i in range(n_states):
            X_smoothed = savgol_filter(X[:, i], self.sg_window, self.sg_order)
            dX[:, i] = np.gradient(X_smoothed, dt)
        return dX

    def _create_polynomial_features(self, X, fit=False):
        """Create polynomial feature library (or use reduced if implemented)."""
        if self.use_reduced_features:
            # Implement if needed, else fallback to polynomial
            pass
        if fit or self.poly is None:
            self.poly = PolynomialFeatures(degree=self.poly_degree, include_bias=False)
            Theta = self.poly.fit_transform(X)
            # Generate feature names
            n_features = X.shape[1]
            from itertools import combinations_with_replacement
            names = []
            for d in range(1, self.poly_degree + 1):
                for combo in combinations_with_replacement(range(n_features), d):
                    names.append(' * '.join([f'x{i+1}' for i in combo]))
            self.feature_names = names
        else:
            Theta = self.poly.transform(X)
        return Theta

    def fit(self, X, dt, verbose=True):
        """
        Train the stacking ensemble.
        Steps:
        1. K-Fold split (K=5).
        2. For each fold, train M=10 Lasso models on the training portion.
        3. Use each trained model to predict on ALL training samples, accumulating
           an (n_samples, n_states, K*M) meta-feature tensor.
        4. Train a Ridge meta-model per state on these K*M meta-features.
        """
        n_samples, n_states = X.shape
        dX = self._compute_derivative_sg(X, dt)
        Theta = self._create_polynomial_features(X, fit=True)
        n_features = Theta.shape[1]

        # Prepare storage for meta-features
        meta_predictions = np.zeros((n_samples, n_states, self.n_estimators))

        kf = KFold(n_splits=self.n_folds, shuffle=True, random_state=42)
        model_idx = 0

        for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
            X_train_fold = X[train_idx]
            Theta_train_fold = Theta[train_idx]
            dX_train_fold = dX[train_idx]

            # Train M base models on this fold's training data
            for m in range(self.n_models_per_fold):
                coeffs = []
                for j in range(n_states):
                    lasso = Lasso(alpha=self.threshold, max_iter=10000,
                                  random_state=fold * self.n_models_per_fold + m)
                    lasso.fit(Theta_train_fold, dX_train_fold[:, j])
                    coef = lasso.coef_
                    # Hard threshold
                    coef[np.abs(coef) < self.threshold] = 0
                    coeffs.append(coef)
                coeffs = np.array(coeffs)   # shape (n_states, n_features)
                self.base_models.append(coeffs)

                # Predict on the ENTIRE training set to get meta-features
                pred_all = Theta @ coeffs.T   # (n_samples, n_states)
                meta_predictions[:, :, model_idx] = pred_all

                model_idx += 1

        # Now meta_predictions has shape (n_samples, n_states, K*M)
        # Train a Ridge meta-model for each state
        self.meta_models = []
        for j in range(n_states):
            X_meta = meta_predictions[:, j, :]   # (n_samples, K*M)
            y_meta = dX[:, j]
            meta_model = Ridge(alpha=self.meta_alpha)
            meta_model.fit(X_meta, y_meta)
            self.meta_models.append(meta_model)

        if verbose:
            print(f"Trained {len(self.base_models)} base models "
                  f"({self.n_folds} folds x {self.n_models_per_fold} models).")

        return self

    def predict_derivative(self, X):
        """
        Predict derivatives using the stacked ensemble.
        For each input state, get predictions from all K*M base models, then
        feed these as meta-features to the Ridge meta-models.
        """
        if X.ndim == 1:
            X = X.reshape(1, -1)
        Theta = self._create_polynomial_features(X, fit=False)
        n_samples = X.shape[0]
        n_states = len(self.meta_models)
        n_models = len(self.base_models)

        # Collect base model predictions
        base_preds = np.zeros((n_samples, n_states, n_models))
        for i, coeffs in enumerate(self.base_models):
            base_preds[:, :, i] = Theta @ coeffs.T

        # Final prediction through meta-models
        final_pred = np.zeros((n_samples, n_states))
        for j in range(n_states):
            final_pred[:, j] = self.meta_models[j].predict(base_preds[:, j, :])

        return final_pred

    def get_final_coefficients(self, threshold=0.1):
        """
        Optional: Combine base model coefficients using meta-model weights
        to obtain a single sparse coefficient matrix per state.
        This is valid because prediction is linear: sum(alpha_i * (Theta * B_i)) = Theta * sum(alpha_i * B_i).
        """
        n_states = len(self.meta_models)
        n_features = self.base_models[0].shape[1]
        final_coeffs = np.zeros((n_states, n_features))
        for j in range(n_states):
            weights = self.meta_models[j].coef_   # (K*M,)
            for k, w in enumerate(weights):
                if k < len(self.base_models):
                    final_coeffs[j] += w * self.base_models[k][j]
            # Optional hard threshold to enforce sparsity
            final_coeffs[j][np.abs(final_coeffs[j]) < threshold] = 0
        return final_coeffs